# Autoship Nudge Promo Incentive — Power Analysis (Autoship Adoption Rate, 2-Cell Design, Observed Allocation Rate)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes a standard 2-cell A/B version of the experiment for its primary metric, **Autoship Adoption Rate**, using the experiment's own live allocation data to measure daily volume — rather than a historical estimate of the broader eligible population.

## Population
Manual clients — i.e., not already enrolled in Autoship — who completed First Fix checkout with a **Buy 1+** keep rate (kept at least one item). Buy 0 clients always see the BAU Quick Fix experience with no Autoship nudge and are out of scope for this comparison, per the PRD.

## Design: 2-cell test, single comparison
Eligible clients are randomized into 2 cells at a 50/50 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment | Autoship nudge + promo billboard | 10% off next eligible Fix |

A single pairwise comparison is planned: **Treatment vs. Control**, measuring the combined effect of introducing the Autoship nudge together with the promo incentive, relative to today's BAU experience. Because only one comparison is planned against the family-wise error budget, no multiple-comparison correction is needed here — sizing uses the initial `alpha = 0.05` directly.

## One-sided test
For Autoship Adoption Rate, a **flat** result (no lift from the nudge+promo experience) and a **negative** result (the nudge+promo experience underperforms BAU) lead to the identical rollout decision: do not roll out the nudge+promo experience, continue with the BAU Quick Fix flow. There is no decision on the table that requires distinguishing "no effect" from "a harmful effect" — so this sizing is **one-sided**, powered only to detect a positive lift from the nudge+promo experience over BAU.

## Why this notebook measures volume from the live allocation log, not from eligibility criteria
The population above describes who *qualifies* for this test. It does not describe who is actually *randomized* — that happens later in the flow, at the moment a client clicks "Schedule a Quick Fix," which is when the Autoship nudge (and therefore the Control/Treatment split) is actually shown. A client who meets the eligibility criteria but never reaches that click is never randomized at all, so estimating daily volume from the eligibility criteria alone risks overstating the true rate at which clients enter the test.

No dedicated tracking flag exists for that specific click, and the closest available tracking event has coverage far too sparse to trust as a stand-alone signal (well under half of what can be independently confirmed through other means) — so historical data cannot reliably answer "how many eligible clients per day actually reach the click."

Once an experiment is live, though, this stops being a proxy problem: **every row in the experiment's allocation log is, by construction, a client who was actually randomized** — a client cannot appear there without having reached the allocation trigger. This notebook uses that property directly: it measures daily volume from the experiment's own live allocation records, rather than reconstructing or approximating it from historical eligibility data. The Autoship Adoption Rate baseline itself still comes from historical data, since that rate needs a matured, multi-month cohort to estimate reliably and the live experiment is too new to supply that on its own.

## Metric definition
**Autoship Adoption Rate** = share of eligible clients who show a fresh Autoship demand event within a 90-day window following their First Fix checkout.

A client's Autoship history is tracked in `curated.client_pulse_journal`, a daily journal (one row per day any tracked client attribute changes) carrying `last_autoship_demand_ts` — the timestamp of that client's most recent Autoship demand event as of that journal row. A client is counted as **adopted** if, scanning their full journal history, the *earliest* `last_autoship_demand_ts` value that is itself later than their First Fix checkout date falls within 90 days of that checkout.

Two choices in this definition are deliberate:
- **Scanning the full journal history, not a single current-day snapshot.** `last_autoship_demand_ts` can reset when a client's Autoship subscription is fully cancelled; reading only today's value would silently drop clients who adopted and later cancelled. Scanning the full history for the earliest post-First-Fix value is robust to that reset.
- **Requiring the demand timestamp to be strictly *after* First Fix checkout, not merely populated.** A meaningful share of clients carry a pre-existing Autoship demand timestamp that predates their First Fix entirely (e.g., a historical Autoship enrollment unrelated to this test's post-checkout nudge). Requiring the timestamp to fall after checkout excludes that population instead of miscounting them as adopters.

**Caveat:** `last_autoship_demand_ts` reflects when a client's Autoship subscription was (re-)created, not each individual shipment under that subscription, so it is an opt-in-*adjacent* signal — one step removed from a literal "clicked opt-in" event log. It also cannot confirm that a given adoption was caused specifically by the post-First-Fix nudge, as opposed to some other Autoship enrollment channel — a limitation shared by any warehouse-derived adoption proxy, not unique to this one.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event to resolve

# The experiment allocates through two plans: a QA plan used for internal validation traffic,
# and the primary plan used for real client allocation. Only the primary plan is used below.
PRIMARY_PLAN_ID = '12bbfac2-30c0-4e39-860b-77240c23bb8f'

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05  # single comparison: Treatment vs. Control, no Bonferroni adjustment needed
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive lift over BAU changes the rollout decision
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05]  # relative lift on Autoship Adoption Rate, Treatment vs. Control

## Step 1 — Autoship Adoption Rate baseline

The eligible population is identified from `curated.merch_sales_and_feedback`: each client's earliest `fix_number = 1` shipment, gated to Manual (`autoship_or_manual = 'manual'`) + Buy 1+ (at least one item kept), with a 90-day maturation cutoff so the demand-event read has had time to resolve. Adoption is then read from `curated.client_pulse_journal`'s full history, per the metric definition above. This baseline is unaffected by the allocation-volume question above — it estimates the *rate* at which a client adopts, not how many clients enter the test per day.

In [2]:
baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN f.first_fresh_demand_ts IS NOT NULL
              AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS adopted_autoship
    FROM eligible e
    LEFT JOIN fresh_demand f ON f.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(adopted_autoship) AS n_adopted,
    CAST(SUM(adopted_autoship) AS DOUBLE) / COUNT(*) AS autoship_adoption_rate
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_adopted,autoship_adoption_rate
0,2026-06-01,5,1463,402,0.274778
1,2026-05-01,31,9549,2396,0.250916
2,2026-04-01,30,10258,2427,0.236596
3,2026-03-01,31,10857,2350,0.216450
4,2026-02-01,28,8799,2106,0.239345
5,2026-01-01,31,10366,2699,0.260370
6,2025-12-01,31,8915,2983,0.334605
7,2025-11-01,30,7248,1462,0.201711
8,2025-10-01,31,9201,1581,0.171829
9,2025-09-01,30,9273,1471,0.158633


In [3]:
# Reference month = most recent calendar month fully past the 90-day maturation cutoff as of this run.
REFERENCE_MONTH = '2026-04-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['autoship_adoption_rate'][0])

print(f"BASELINE_RATE = {BASELINE_RATE:.4f}")
ref.T

BASELINE_RATE = 0.2366


,0
month,2026-04-01
days_observed,30
n_eligible,10258
n_adopted,2427
autoship_adoption_rate,0.236596


**Reference month choice:** the most recent month with every calendar day already past the 90-day maturation cutoff — more recent months are only partially mature (fewer of their days have had 90 days to produce an adoption read yet), so their rates aren't yet comparable to a full month.

## Step 2 — Observed daily allocation rate from the live experiment

The experiment's allocation log records one row per client at the moment they are actually randomized — which only happens once they reach the "Schedule a Quick Fix" click. This step measures the daily rate directly from that log, over the full period the experiment has been running so far.

In [4]:
allocation_query = f"""--sql
SELECT
    MIN(event_ts) AS first_allocation_ts,
    CURRENT_TIMESTAMP AS observation_ts,
    DATE_DIFF('hour', MIN(event_ts), CURRENT_TIMESTAMP) AS hours_observed,
    COUNT(DISTINCT rand_unit_value) AS n_allocated_total,
    COUNT(DISTINCT CASE WHEN cell_id = 1 THEN rand_unit_value END) AS n_control,
    COUNT(DISTINCT CASE WHEN cell_id = 2 THEN rand_unit_value END) AS n_treatment
FROM ab.current_allocations
WHERE plan_id = '{PRIMARY_PLAN_ID}'
"""

allocation_df = query(allocation_query)
allocation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,first_allocation_ts,observation_ts,hours_observed,n_allocated_total,n_control,n_treatment
0,2026-08-12 20:57:06.866,2026-09-03 18:19:40.907 UTC,525,1623,831,792


In [5]:
days_observed = float(allocation_df['hours_observed'][0]) / 24.0
DAILY_ELIGIBLE = float(allocation_df['n_allocated_total'][0]) / days_observed

print(f"days_observed = {days_observed:.2f}  |  DAILY_ELIGIBLE (observed) = {DAILY_ELIGIBLE:.1f} clients/day")

days_observed = 21.88  |  DAILY_ELIGIBLE (observed) = 74.2 clients/day


**Reading this:** the observed daily allocation rate reflects the current run rate of the live experiment, not a projection. If the experience is still ramping up (e.g., a phased engineering rollout), this rate will understate the eventual steady-state volume — this figure should be refreshed periodically as more allocation history accumulates, and re-checked once the rollout is known to have reached full ramp.

## Step 2a — Planned client exposure to Treatment and Control

Using the observed daily allocation rate from Step 2 and the experiment's 50/50 split, this section
translates that rate into how many clients enter each arm per day, per week, and over the full
experiment — at a +4% relative MDE, the same design checkpoint carried through the rest of this
notebook. Every client allocated to the Treatment arm is exposed to the Autoship nudge + promo;
every client allocated to the Control arm sees the existing BAU Quick Fix experience with no nudge
and no promo.

In [6]:
# Same +4% relative-MDE checkpoint used as this notebook's headline design point.
EXPOSURE_TARGET_REL_MDE = 0.04

exposure_res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[EXPOSURE_TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)
n_per_arm_at_completion = int(list(exposure_res.values())[0]['n_treatment'])
days_to_completion = int(np.ceil(n_per_arm_at_completion / (DAILY_ELIGIBLE / N_ARMS)))

daily_per_arm = DAILY_ELIGIBLE / N_ARMS
weekly_per_arm = daily_per_arm * 7

exposure_table = pd.DataFrame(
    {
        'Control (no nudge, no promo)': [round(daily_per_arm, 1), round(weekly_per_arm, 1), n_per_arm_at_completion],
        'Treatment (nudge + promo)': [round(daily_per_arm, 1), round(weekly_per_arm, 1), n_per_arm_at_completion],
    },
    index=['Per day', 'Per week', f'Full experiment (~{days_to_completion} days at +{EXPOSURE_TARGET_REL_MDE:.0%} MDE)'],
)
exposure_table


,"Control (no nudge, no promo)",Treatment (nudge + promo)
Per day,37.1,37.1
Per week,259.7,259.7
Full experiment (~682 days at +4% MDE),25276.0,25276.0


## Step 3 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment vs. Control); `n_treatment` is read as the **per-arm** requirement. Each of the 2 arms accrues `DAILY_ELIGIBLE / 2` clients per day under the 50/50 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE / 2)`.

In [7]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_2arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- Autoship Adoption Rate, Treatment vs. Control (baseline=23.7%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+2%,0.241328,100428,200856,2708,386.9
1,+3%,0.243694,44786,89572,1208,172.6
2,+4%,0.24606,25276,50552,682,97.4
3,+5%,0.248426,16231,32462,438,62.6


**Reading this:** required duration at every MDE checkpoint is dramatically longer than a sizing exercise based on the broader eligibility criteria would suggest, because the observed live allocation rate is a small fraction of that broader population's daily volume. This is exactly the gap the eligibility-based estimate could not see — a historical query can only describe who *qualifies*, not who the live system is actually admitting into the test per day. No harm/guardrail grid is computed here: sizing for a one-sided positive MDE does not symmetrically size for detecting harm, and downside risk on this metric is out of scope for this sizing exercise (margin risk is covered qualitatively via the Experiment Design doc's Risk section and Decision Matrix, not powered here).

## Step 4 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 4% relative lift** on Autoship Adoption Rate (Treatment vs. Control) — the second-largest value in the Step 3 grid.

In [8]:
TARGET_REL_MDE = 0.04  # placeholder: 4% relative lift on Autoship Adoption Rate, Treatment vs. Control (second-largest value in the Step 3 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate (Treatment vs. Control)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship',
    'Baseline Value': f"{BASELINE_RATE:.1%} ({REFERENCE_MONTH[:7]}, {MATURATION_DAYS}-day matured cohort)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.1f} / day (observed live allocation rate, primary plan, {days_observed:.1f} days observed)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (2 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate (Treatment vs. Control)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship"
Baseline Value,"23.7% (2026-04, 90-day matured cohort)"
Daily Eligible Volume,"74.2 / day (observed live allocation rate, primary plan, 21.9 days observed)"
Minimum Detectable Effect,+4% relative (0.237 -> 0.246)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"25,276"


## Bottom line

- **This notebook measures daily volume from the experiment's own live allocation log**, not from historical eligibility criteria — a client only appears in that log once they are actually randomized, so this figure is guaranteed accurate rather than approximated.
- **The observed daily allocation rate is far lower than a historical eligibility-based estimate would suggest.** That gap reflects real drop-off between qualifying for the test and actually reaching the allocation trigger, and possibly an ongoing rollout ramp — either way, it's a live measurement, not a modeling assumption.
- **Required duration at every MDE checkpoint in Step 3 is dramatically longer** than sizing on the broader eligible population would imply, because the true daily rate of randomized clients is much smaller than the population that merely qualifies for the test.
- **This observed rate should be refreshed as more allocation history accumulates** — a short observation window can understate or overstate the eventual steady-state rate, particularly during an early or ramping rollout.
- At a **4% relative lift** (the placeholder MDE above), the experiment needs the sample size and duration shown in Step 4.